# STAC Item for NOAA CDR NDVI — Virtual Icechunk on Cloudflare R2

Creates a [STAC](https://stacspec.org/) Item describing the NOAA CDR NDVI virtual Icechunk
store that lives on Cloudflare R2 (`ndvi-cdr-icechunk`).  
Follows the pattern from the [DSE Virtual Zarr Workshop](https://nasa-impact.github.io/dse-virtual-zarr-workshop/examples/03_STAC_Collection_for_Virtual_Icechunk_Store.html).

**Steps**
1. Open the Icechunk store from R2 and read dataset metadata
2. Build a `pystac.Item` template
3. Populate spatial / temporal extents with `xstac`
4. Add Icechunk asset with `zarr`, `virtual-assets`, and `storage` extension fields
5. Validate and save to `ndvi_cdr_stac_item.json`
6. Round-trip: reconstruct the store from the saved item JSON using `icechunk` + `xarray`

In [ ]:
import json
import datetime
import warnings

import xarray as xr
import icechunk
import pystac
import xstac

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## 1. Open the Icechunk store from R2

The store is publicly accessible — no credentials needed.
We pin the **snapshot ID** so the STAC item always refers to a reproducible version of the store.

In [ ]:
snapshot_id = "TF8SVRDCRCC58SKCVK20"

storage = icechunk.r2_storage(
    endpoint_url="https://r2-pub.openscicomp.io",
    bucket="ndvi-cdr-icechunk",
    prefix=None,
    anonymous=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://noaa-cdr-ndvi-pds/",
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    ),
)

credentials = icechunk.containers_credentials({
    "s3://noaa-cdr-ndvi-pds/": icechunk.s3_credentials(anonymous=True)
})

repo = icechunk.Repository.open(
    storage, config, authorize_virtual_chunk_access=credentials
)

session = repo.readonly_session(snapshot_id=snapshot_id)
ds = xr.open_zarr(session.store, consolidated=False, chunks=None)
ds

## 2. Build the STAC Item template

A STAC Item covers a single "feature" — here the 5-day NDVI CDR slab.
We use `datetime=None` and provide `start_datetime` / `end_datetime` in
`properties` because the item spans multiple days.

In [ ]:
item_id = f"noaa-cdr-ndvi-icechunk@{snapshot_id}"

# Derive temporal bounds directly from the dataset
start_dt = datetime.datetime.fromisoformat(str(ds.time.min().values))
end_dt   = datetime.datetime.fromisoformat(str(ds.time.max().values))

# Global extent placeholder — xstac will refine from the dataset coordinates
bbox = [-180.0, -90.0, 180.0, 90.0]
geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-180.0, -90.0], [180.0, -90.0],
        [180.0,  90.0],  [-180.0,  90.0],
        [-180.0, -90.0],
    ]]
}

template = pystac.Item(
    id=item_id,
    geometry=geometry,
    bbox=bbox,
    datetime=None,   # multi-day range → use start/end instead
    properties={
        "start_datetime": start_dt.isoformat() + "Z",
        "end_datetime":   end_dt.isoformat()   + "Z",
        "title": ds.attrs.get("title", "NOAA CDR NDVI — Virtual Icechunk"),
        "description": (
            "Virtual Zarr references to the NOAA Climate Data Record (CDR) of AVHRR NDVI "
            "v5 stored as an Icechunk repository on Cloudflare R2. Chunk data remains on "
            "the original AWS S3 source bucket `noaa-cdr-ndvi-pds`. "
            "You can open this dataset in xarray:\n"
            "```python\n"
            "import requests, xarray as xr, icechunk\n"
            "item = requests.get(\"<STAC_ITEM_URL>\").json()\n"
            "asset = next(v for v in item[\"assets\"].values()\n"
            "             if v.get(\"type\") == \"application/vnd.zarr+icechunk\")\n"
            "scheme = item[\"storage:schemes\"][asset[\"storage:refs\"][0]]\n"
            "storage = icechunk.r2_storage(\n"
            "    endpoint_url=scheme[\"endpoint_url\"],\n"
            "    bucket=scheme[\"bucket\"],\n"
            "    anonymous=True,\n"
            ")\n"
            "repo = icechunk.Repository.open(storage)\n"
            "session = repo.readonly_session(snapshot_id=asset[\"icechunk:snapshot_id\"])\n"
            "ds = xr.open_zarr(session.store, consolidated=False)\n"
            "```"
        ),
    },
    stac_extensions=[
        "https://stac-extensions.github.io/zarr/v1.1.0/schema.json",
        "https://stac-extensions.github.io/datacube/v2.2.0/schema.json",
        "https://stac-extensions.github.io/virtual-assets/v1.0.0/schema.json",
        "https://stac-extensions.github.io/version/v1.2.0/schema.json",
    ],
)
template

## 3. Populate spatial / temporal extents with xstac

`xstac.xarray_to_stac` inspects the dataset coordinates and fills in
`bbox`, `geometry`, and the `cube:dimensions` / `cube:variables` datacube fields.

In [ ]:
item = xstac.xarray_to_stac(
    ds,
    template,
    temporal_dimension="time",
    x_dimension="longitude",
    y_dimension="latitude",
    reference_system=4326,
    validate=False   # we validate explicitly after adding assets
)

print("bbox:", item.bbox)
print("geometry type:", item.geometry["type"])
item

## 4. Add the Icechunk asset

In [ ]:
storage_schemes = {
    "r2-osc-pub": {
        "type": "aws-s3",
        "platform": "https://r2-pub.openscicomp.io",
        "bucket": "ndvi-cdr-icechunk",
        "region": "not-used",
        "endpoint_url": "https://r2-pub.openscicomp.io",
        "anonymous": True,
    },
}

item.extra_fields["storage:schemes"] = storage_schemes

In [ ]:
out_path = "ndvi_cdr_stac_item.json"

icechunk_key = f"ndvi-cdr-icechunk@{snapshot_id}"

item.add_asset(
    icechunk_key,
    pystac.Asset(
        href="s3://ndvi-cdr-icechunk/",
        title="NOAA CDR NDVI — Virtual Icechunk Store (R2)",
        media_type="application/vnd.zarr+icechunk",
        roles=["data", "references", "virtual", "latest-version"],
        extra_fields={
            "zarr:consolidated": False,
            "zarr:zarr_format": 3,
            "icechunk:snapshot_id": snapshot_id,
            "storage:refs": ["r2-osc-pub"],
        },
    ),
)

item.assets

## 5. Validate and save

In [ ]:
item.validate()

with open(out_path, "w") as f:
    json.dump(item.to_dict(), f, indent=2)

print(f"Saved → {out_path}")
print(json.dumps(item.to_dict(), indent=2))

## 5b. Upload STAC item JSON to R2

Make the item discoverable at a public HTTP URL, then patch that URL back into the description's code snippet.

In [ ]:
import os, boto3
from dotenv import load_dotenv

load_dotenv(f'{os.environ["HOME"]}/dotenv/r2-osc-pub.env')

r2_write_endpoint = os.environ["ENDPOINT_URL"]
r2_write_bucket   = "osc-pub"
s3_key            = f"stac/ndvi/{out_path}"

s3 = boto3.client(
    "s3",
    endpoint_url=r2_write_endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name="auto",
)

# First upload
s3.upload_file(out_path, r2_write_bucket, s3_key, ExtraArgs={"ContentType": "application/json"})

public_stac_url = f"https://r2-pub.openscicomp.io/{s3_key}"
print(f"STAC item uploaded → {public_stac_url}")

# Patch <STAC_ITEM_URL> in the saved JSON with the actual public URL
with open(out_path) as f:
    item_dict = json.load(f)

item_dict["properties"]["description"] = item_dict["properties"]["description"].replace(
    "<STAC_ITEM_URL>", public_stac_url
)

with open(out_path, "w") as f:
    json.dump(item_dict, f, indent=2)

# Re-upload with patched URL
s3.upload_file(out_path, r2_write_bucket, s3_key, ExtraArgs={"ContentType": "application/json"})
print(f"Re-uploaded with actual URL → {public_stac_url}")

## 6. Round-trip: open dataset from saved STAC item JSON

Load the saved item JSON and reconstruct the Icechunk store directly from its fields.
`xpystac` doesn't support custom S3-compatible endpoints (like Cloudflare R2), so we
extract the storage configuration manually and open the store with `icechunk` + `xarray`.

In [ ]:
loaded_item = pystac.Item.from_file(out_path)

# Pull fields from the icechunk asset
ic_asset = next(
    a for a in loaded_item.assets.values()
    if a.media_type == "application/vnd.zarr+icechunk"
)
snap_id  = ic_asset.extra_fields["icechunk:snapshot_id"]
ref      = ic_asset.extra_fields["storage:refs"][0]
scheme   = loaded_item.extra_fields["storage:schemes"][ref]

bucket   = scheme["bucket"]
endpoint = scheme["endpoint_url"]
prefix   = ic_asset.href.split(f"{bucket}/")[1] or None

print(f"bucket={bucket}, prefix={prefix}, snapshot={snap_id}")

storage2 = icechunk.r2_storage(
    endpoint_url=endpoint,
    bucket=bucket,
    prefix=prefix,
    anonymous=True,
)

repo2    = icechunk.Repository.open(storage2)
session2 = repo2.readonly_session(snapshot_id=snap_id)
ds2      = xr.open_zarr(session2.store, consolidated=False, chunks=None)
ds2